In [ ]:
%load_ext autoreload
%autoreload 2

# DynaSurvCausalOnline Model Validation

## Objectives
1. **Factual Consistency**: Calibration and discriminative performance (C-index, Brier Score).
2. **Counterfactual Stability**: Pairwise constraints and temporal coherence.
3. **Clinical Plausibility**: Stratified analysis by biomarkers.
4. **Policy Evaluation**: Regret analysis.

In [ ]:
from collections import defaultdict
from copy import deepcopy
from numbers import Number

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from lifelines import KaplanMeierFitter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sksurv.nonparametric import kaplan_meier_estimator
from tqdm import tqdm

from CausalSurv.data.datamodule_cv import ESMEOnlineDataModuleCV
from CausalSurv.evaluation.evaluator import DynasurvEvaluator
from CausalSurv.model import DynaSurvCausalOnline

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
def single_batchify_trainingloader(data_module):

    dm = deepcopy(data_module)

    dm.batch_size = len(dm.ESMEDataset)
    dm.setup()
    train_loader = dm.train_dataloader()

    return train_loader

In [ ]:
# Update these paths as needed or load from a central config
MODEL_PATH = "/Users/malek/TheLAB/DynaSurv/models/HR+HER2-/4lines/06032026_141113_seed_2021779468/checkpoints/dynaSurvCausalOnline-epoch=164-val_loss= 1.1227.ckpt"
DATA_PATH = "/Users/malek/TheLAB/DynaSurv/data"
CONFIG_PATH = "/Users/malek/TheLAB/DynaSurv/configs/config_train.toml"
SPLIT_SEED = 2021779468
SUBTYPE = "HR+HER2-"
N_LINES = 4
BATCH_SIZE = 256

##  Load Model and Data

In [ ]:
# # Load Model
print(f"Loading model from {MODEL_PATH}...")
model = DynaSurvCausalOnline.load_from_checkpoint(
    MODEL_PATH, map_location=torch.device("cpu")
)
model.eval()
model.freeze()
print("Model loaded.")

# Load Data
print("Loading data...")
data_module = ESMEOnlineDataModuleCV(
    data_dir=DATA_PATH,
    subtype=SUBTYPE,
    n_intervals=100 - 1,
    n_lines=N_LINES,
    batch_size=BATCH_SIZE,
    final_training=True,
    num_workers=4,
    split_seed=SPLIT_SEED,
)
data_module.prepare_data()
data_module.setup()
print("Data loaded.")

evaluator = DynasurvEvaluator(model, data_module)
print("Evaluator setup")

## Treatment distribution

In [ ]:
train_dataset = data_module.cv_dataset
traetment_dics = data_module.treatment_dict
pat_idx = []
for idx in range(len(train_dataset)):
    *a , pat_id = train_dataset[idx]
    pat_idx.append(pat_id)

df = data_module.df_dynamic

# top_categories = df["T_treatment_category"].value_counts().nlargest(7).index
# df["T_treatment_category"] = df["T_treatment_category"].apply(
#     lambda x: x if x in top_categories else "OTHER"
# )

df_counts = (
    df.loc[df["lineid"] <= 4].groupby(["lineid", "T_treatment_category"]).size().reset_index(name="count")
)

pivot_df = df_counts.pivot(
    index="lineid", columns="T_treatment_category", values="count"
).fillna(0)



pivot_df

In [ ]:
pivot_df.plot(
    kind="bar",
    stacked=True,
    width=0.75,
    edgecolor="white",  # White border between segments for "clean" look
    linewidth=0.5,
)

## Factual Consistency Check

In [ ]:
test_loader = data_module.test_dataloader()
interval_bounds = model.interval_bounds.to("cpu")
res = evaluator.test_model()

## Predicted survival:

In [ ]:
def plot_predicted_survival_sample(test_dataloader, n_samples, ncols=5):
    (XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(test_dataloader))
    
    sample_idx = np.random.choice(np.arange(XPd.shape[0]), n_samples)
    
    nrows = (n_samples + ncols -1) // ncols
    fig, ax = plt.subplots(nrows, ncols, figsize=(7 * n_samples, 6 * nrows))
    
    surv = model.predict_discrete_survival(XPd, X_static, gather=True, factual_idx = treatment_indices)
    surv_np = surv.numpy()
    N_LINES = surv_np.shape[1]
    clr = plt.colormaps["tab10"].colors[:N_LINES]
    for c, pat_idx in enumerate(sample_idx):
        i = c // ncols
        j = c % ncols
        for line in range(N_LINES):
            mask_line = mask[:,line].bool()
            true_times, true_probs = kaplan_meier_estimator(event.squeeze()[mask_line, line].bool(),
                                                            time.squeeze()[mask_line,line])
            if not mask[pat_idx, line].bool().any():
                continue
            if nrows == 1:
                ax[j].step(x=interval_bounds,y=surv_np[pat_idx,line], label=f"line {line}", color=clr[line])
                ax[j].set_title(f"Patient {patient_id[pat_idx]}")
                ax[j].step(x=true_times,y=true_probs, label=f"line {line} KM", color=clr[line], linestyle="--")
                ax[j].set_xlim(0,50)
                ax[j].legend()
            else:
                ax[i,j].step(x=interval_bounds,y=surv_np[pat_idx,line], label=f"line {line} KM", color=clr[line])
                ax[i,j].set_title(f"Patient {patient_id[pat_idx]}")
                ax[i,j].step(x=true_times,y=true_probs, label=f"line {line}", color=clr[line], linestyle="--")
                # ax[i,j].set_xlim(0,50)
                ax[i,j].legend()
    plt.suptitle("Sample survival predictions")
    plt.tight_layout()
    plt.show()
  
plot_predicted_survival_sample(data_module.test_dataloader(), 5)


In [ ]:
def plot_times_distribution(test_dataloader):
    (XPd, *_, time, _, mask, _) = next(iter(test_dataloader))
    time = time.numpy()
    mask = mask.numpy()
    N_LINES = XPd.shape[1]
    fig, ax = plt.subplots(nrows=1,ncols=N_LINES,figsize=(4 * N_LINES, 3))
    for line in range(N_LINES):
        valid_mask = mask[:, line] == 1
        if not valid_mask.any():
            continue
        t = time[valid_mask, line]
        med, q90 = np.percentile(t, [50, 90])

        ax[line].hist(t, bins=50)
        ax[line].axvline(med, color="green", lw=2, label=f"Median = {med:.1f}")
        ax[line].axvline(q90, color="red", lw=2, label=f"90% = {q90:.1f}")
        ax[line].set_title(f"Line {line+1}")
        ax[line].set_xlabel("Time")
        ax[line].legend(fontsize=8)
        ax[line].set_ylim(0,900)

    fig.suptitle("Distribution of event times by treatment line", fontsize=14)
    plt.tight_layout()
    plt.show()
    
plot_times_distribution(test_loader)

In [ ]:
def plot_km_per_line(model, test_dataloader):
    (XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(test_dataloader))
    
    fig, ax = plt.subplots(nrows=1, ncols=3, figsize = (15, 3))
    N_LINES = XPd.shape[1]
    for line in range(N_LINES):
        valid_mask = mask[:,line]==1
        e_line = model.train_events[line].ravel()
        t_line = model.train_times[line].ravel()

        surv_times, surv_probs = kaplan_meier_estimator(event=event.squeeze()[valid_mask, line].bool(), #type: ignore
                                              time_exit=time.squeeze()[valid_mask, line])
        train_times, train_prob = kaplan_meier_estimator(e_line,t_line) # type: ignore 
        cens_times, cens_prob = kaplan_meier_estimator(~e_line,t_line)
        
        ax[0].step(surv_times, surv_probs, label=f"Line {line}")
        ax[0].legend()
        ax[0].set_title("test")

        ax[1].step(train_times, train_prob, label=f"Line {line}")
        ax[1].legend()
        ax[1].set_title("train")

        ax[2].step(cens_times, cens_prob, label=f"Line {line}")
        ax[2].legend()
        ax[2].set_title("censoring")

    plt.tight_layout()
    plt.suptitle("Kaplan Meier")
    plt.show()
        
plot_km_per_line(model, data_module.test_dataloader())

In [ ]:
bs_res = evaluator.brier_score([48, 23, 15, 13], plot=True)

## Calibration Plot

### Overall calibration

In [ ]:
ece_res = evaluator.line_calibration_error(eval_time=[6, 12, 18, 24], plot=True)

### Treatment wise calibration

In [ ]:
treatment_ece_res = evaluator.treatment_calibration_error(eval_time=[6, 12, 18], plot=True)

## Adversarial unconfounding
Check if the learned representation is not predictive of the treatment

In [ ]:
(XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(test_loader))
model.compute_treatment_prediction_auc(XPd, X_static, treatment_indices, mask)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

probabilities = {}
accuracies = {}
auc_scores = {}
dummy_auc_scores = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        h, c, p = model._init_lstm_states(X_static, XPd.device)

        for line in range(XPd.shape[1]):
            _, h, c, p = model.lstm(XPd[:, line, :], (h, c, p))

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].cpu().numpy()
            y = treatment_indices[mask_line, line].cpu().numpy()
            # print(np.unique(y, return_counts=True))
            # plot_umap_deconfounding(X, y, title=f"UMAP Projection - Line {line + 1}")

            unique_classes, counts = np.unique(y, return_counts=True)
            valid_classes = unique_classes[counts >= 5]
            valid_indices = np.isin(y, valid_classes)
            X = X[valid_indices]
            y = y[valid_indices]

            # print(X.shape, y.shape)
            if len(np.unique(y)) < 2:
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_train, y_train)
            acc = rfc.score(X_test, y_test)
            rfc_probs = rfc.predict_proba(X_test)

            dummy = DummyClassifier(strategy="stratified", random_state=42)
            dummy.fit(X_train, y_train)
            dummy_probs = dummy.predict_proba(X_test)

            # print(f"random forest probs:{rfc_probs.shape}, y_test.shape:{y_test.shape}")
            # print(f"dummy dummy_probs.shape:{dummy_probs.shape}, y_test.shape:{y_test.shape}")

            rfc_auc = roc_auc_score(
                y_test,
                rfc_probs,
                multi_class="ovr",
                average="macro",
                labels=rfc.classes_,
            )
            dummy_auc = roc_auc_score(
                y_test,
                dummy_probs,
                multi_class="ovr",
                average="macro",
                labels=dummy.classes_,
            )

            # Store results
            accuracies[line] = acc
            auc_scores[line] = rfc_auc
            dummy_auc_scores[line] = dummy_auc
            probabilities[line] = rfc_probs

# --- Print Results Table ---
print(f"\n{'Line':<10} | {'RFC Accuracy':<15} | {'RFC AUC':<10} | {'Dummy AUC':<10}")
print("-" * 55)
for line in range(N_LINES):
    if line in accuracies:
        print(
            f"Line {line + 1:<5} | {accuracies[line]:.4f}         | {auc_scores[line]:.4f}     | {dummy_auc_scores[line]:.4f}"
        )
    else:
        print(f"Line {line + 1:<5} | No valid data.")

## Virtual clinical trial

In [ ]:
def run_ai_vs_doctor(dataloader, interval_bounds, eval_time: list | int | float = 50, ):  
    (XPd, X_static, interval_idx, treatment_indices, time, event, mask, patient_id) = batch
    BATCH_SIZE, N_LINES = XPd.shape[:2]
    N_TREATMENTS = torch.unique(treatment_indices).numel()

    if isinstance(eval_time, Number):
        eval_time = [eval_time] * N_LINES
    assert isinstance(eval_time, list)
    if len(dataloader) > 1:
        print("function expects a single batch, defaulting to the first")

    concordant_group = defaultdict(lambda: defaultdict(list))
    discordant_group = defaultdict(lambda: defaultdict(list))
    rmst_list = []

    disc_survival = model.predict_discrete_survival(XPd, X_static, gather=False)
    dt = interval_bounds[1] - interval_bounds[0]

    rmst = torch.cumsum(disc_survival, dim=3) * dt #(batch, n_lines, n_treatments, n_intervals)
    rmst_idx = torch.bucketize(torch.tensor(eval_time).view(N_LINES, -1), interval_bounds) #(n_lines, 1)
    gather_idx = rmst_idx.unsqueeze(0).unsqueeze(-1).expand(BATCH_SIZE, N_LINES, N_TREATMENTS, -1)
    rmst_at_horizon = torch.gather(rmst, dim= 3, index=gather_idx).squeeze()

    recommended_treatment = rmst_at_horizon.argmax(dim=2)  # (batch, n_lines)
    observed_treatment = treatment_indices  # (batch, n_lines)

    for line in range(N_LINES):
        valid_mask = mask[:, line].bool()
        if not valid_mask.any():
            continue

        rec = recommended_treatment[valid_mask, line].cpu().numpy()
        obs = observed_treatment[valid_mask, line].cpu().numpy()

        real_t = time[valid_mask, line].cpu().numpy()
        real_e = event[valid_mask, line].cpu().numpy()

        matches = rec == obs

        concordant_group[line]["time"].extend(real_t[matches])
        concordant_group[line]["event"].extend(real_e[matches])

        discordant_group[line]["time"].extend(real_t[~matches])
        discordant_group[line]["event"].extend(real_e[~matches])
    
        rmst_list.append(rmst_at_horizon / eval_time[line])
    return concordant_group, discordant_group, recommended_treatment, observed_treatment, mask

In [ ]:
def _rmst_from_km(kmf):
    t = kmf.survival_function_.index.values
    s = kmf.survival_function_[kmf.label]

    dt = np.diff(np.insert(t, 0, 0))
    rmst = np.cumsum(s * dt)
    return t, rmst


def plot_conc_disc(concordant, discordant):

    N_LINES = len(concordant)

    fig, axes = plt.subplots(2, N_LINES, figsize=(6 * N_LINES, 8), sharex="col")

    if N_LINES == 1:
        axes = np.array(axes).reshape(2, 1)

    for line in range(N_LINES):

        ax_km = axes[0, line]
        ax_rmst = axes[1, line]

        kmf_concordant = KaplanMeierFitter()
        kmf_discordant = KaplanMeierFitter()

        T_conc, E_conc = concordant[line]["time"], concordant[line]["event"]
        T_disc, E_disc = discordant[line]["time"], discordant[line]["event"]

        kmf_concordant.fit(
            T_conc, event_observed=E_conc,
            label=f"Matched strategy (n={len(T_conc)})"
        )

        kmf_discordant.fit(
            T_disc, event_observed=E_disc,
            label=f"Differed strategy (n={len(T_disc)})"
        )

        # --- KM curves ---
        kmf_concordant.plot_survival_function(ax=ax_km, ci_show=True, color="green")
        kmf_discordant.plot_survival_function(ax=ax_km, ci_show=True, color="red")

        ax_km.set_title(f"Line {line+1}")
        ax_km.set_ylabel("Survival Probability")

        # --- RMST curves ---
        t_conc, rmst_conc = _rmst_from_km(kmf_concordant)
        t_disc, rmst_disc = _rmst_from_km(kmf_discordant)

        ax_rmst.plot(t_conc, rmst_conc, color="green", label="Matched strategy")
        ax_rmst.plot(t_disc, rmst_disc, color="red", label="Differed strategy")

        ax_rmst.set_xlabel("Time")
        ax_rmst.set_ylabel("RMST(t)")

        ax_km.legend()
        ax_rmst.legend()

        ax_km.set_xlim(0, 50)
        ax_rmst.set_xlim(0, 50)

    fig.suptitle("Model Recommendation vs. Actual Treatment")
    # plt.tight_layout()
    plt.show()



In [ ]:
conc, disc, recommended_treatment, observed_treatment, mask = run_ai_vs_doctor(test_loader, eval_time=10, interval_bounds=data_module.interval_bounds)
plot_conc_disc(conc, disc)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_treatment_mix(recommended_treatment, observed_treatment, treatment_dict, mask):
    N_LINES = recommended_treatment.shape[1]
    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 6))
    
    cmap = plt.colormaps["tab20"]
    unique_treatments = np.unique(np.concatenate([recommended_treatment, observed_treatment]))
    colors = {t: cmap(i) for i, t in enumerate(unique_treatments)}

    def plot_on_axis(data, mask, axis, title):
        used_labels = set()
        for line in range(N_LINES):
            mask_line = mask[:, line]
            values, counts = np.unique(data[mask_line, line], return_counts=True)
            sort_idx = np.argsort(counts)
            values, counts = values[sort_idx], counts[sort_idx]

            bottom = 0
            for val, cnt in zip(values, counts):
                label = treatment_dict[val] if val not in used_labels else None
                used_labels.add(val)

                axis.bar(
                    line,
                    cnt,
                    bottom=bottom,
                    color=colors[val],
                    label=label
                )
                bottom += cnt

        axis.set_xticks(range(N_LINES))
        axis.set_xticklabels([f"Line {i+1}" for i in range(N_LINES)])
        axis.set_ylabel("Count")
        axis.set_title(title)

    # Plot recommended
    plot_on_axis(recommended_treatment, mask, ax[0], "Recommended Treatment Mix")
    # Plot observed
    plot_on_axis(observed_treatment, mask, ax[1], "Observed Treatment Mix")

    # Only show legend once
    ax[1].legend(loc='upper right', bbox_to_anchor=(1.3, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
plot_treatment_mix(recommended_treatment, observed_treatment, data_module.treatment_dict, mask)

## Treatment strategy robustness:

In [ ]:
# import numpy as np
# import pandas as pd

# def filter_recommended_treatments(recommended_treatment, mask):
#     n_lines = recommended_treatment.shape[1]
#     val, count = [], []
#     for line in range(n_lines):
#         valid_mask = mask[:, line]
#         v, c = np.unique(recommended_treatment[valid_mask,line], return_counts=True)
#         print(v,c)
#         val.append(v)
#         count.append(c)
#     return val, count

# filter_recommended_treatments(recommended_treatment, mask)
# 
# 
# 
# def strategy_robustness(dataloader, interval_bounds, horizons):
#     results = []
#     treatment_mixes = []

#     for h in horizons:
#         conc, disc, recommended_treatment, observed_treatment, mask = run_ai_vs_doctor(dataloader, interval_bounds, eval_time=h)
        
#         values, counts = np.unique(recommended_treatment, return_counts=True)
#         mix = dict(zip(values, counts))

#         results.append({'horizon': h, 'concordant': conc, 'discordant': disc})
#         treatment_mixes.append({'horizon': h, **mix})

#     results_df = pd.DataFrame(results)
#     treatment_mixes_df = pd.DataFrame(treatment_mixes).fillna(0).astype(int)  # fill missing treatments with 0

#     return results_df, treatment_mixes_df

# rest_df, treatment_mix_df  = strategy_robustness(test_loader, data_module.interval_bounds, np.linspace(1, 100, 100))

In [ ]:
treatment_mix_df.rename(columns=data_module.treatment_dict, inplace=True)
treatment_mix_df.plot(figsize=(7,3))

plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), title="Treatment")
plt.xlabel("Horizon")
plt.ylabel("Number of patients")
plt.title("Treatment mix evolution")
plt.tight_layout()  # adjust layout to avoid clipping
plt.show()

## Treatment recommendation analysis

# 🚧🚧🚧🚧🚧🚧🚧

In [ ]:
# from sksurv.metrics import brier_score, cumulative_dynamic_auc
# from sksurv.nonparametric import kaplan_meier_estimator
# import numpy as np
# import matplotlib.pyplot as plt

# def plot_calibration_improved(preds_surv, obs_times, obs_events, time_point,
#                                n_bins=10, strategy='quantile'):
#     """
#     Better calibration plot with more robust binning and error bars.

#     Parameters:
#     - strategy: 'uniform' (equal width) or 'quantile' (equal sample size per bin)
#     """

#     # Remove invalid predictions
#     valid_mask = ~np.isnan(preds_surv) & (preds_surv >= 0) & (preds_surv <= 1)
#     preds_surv = preds_surv[valid_mask]
#     obs_times = obs_times[valid_mask]
#     obs_events = obs_events[valid_mask]

#     if len(preds_surv) < n_bins * 5:
#         print(f"Warning: Only {len(preds_surv)} samples, reducing bins")
#         n_bins = max(3, len(preds_surv) // 10)

#     # Better binning strategy
#     if strategy == 'quantile':
#         # Equal number of samples per bin
#         bin_edges = np.percentile(preds_surv, np.linspace(0, 100, n_bins + 1))
#         bin_edges[-1] += 1e-8  # Ensure last bin includes max value
#     else:
#         # Equal width bins
#         bin_edges = np.linspace(0, 1, n_bins + 1)

#     bin_indices = np.digitize(preds_surv, bin_edges) - 1
#     bin_indices = np.clip(bin_indices, 0, n_bins - 1)  # Handle edge cases

#     calib_pred = []
#     calib_obs = []
#     calib_counts = []
#     calib_se = []  # Standard errors for confidence intervals

#     for i in range(n_bins):
#         idx = bin_indices == i
#         count = idx.sum()

#         if count < 5:  # Skip bins with too few samples
#             continue

#         # Average prediction in bin
#         avg_pred = preds_surv[idx].mean()

#         # Kaplan-Meier estimate at time_point
#         try:
#             time, survival_prob = kaplan_meier_estimator(
#                 obs_events[idx].astype(bool),
#                 obs_times[idx]
#             )

#             # Interpolate at time_point
#             if time_point <= time[0]:
#                 obs_surv = 1.0
#                 se = 0.0
#             elif time_point >= time[-1]:
#                 obs_surv = survival_prob[-1]
#                 # Greenwood's formula for KM variance (simplified)
#                 se = np.sqrt(survival_prob[-1] * (1 - survival_prob[-1]) / count)
#             else:
#                 # Find surrounding time points
#                 idx_after = np.searchsorted(time, time_point)
#                 obs_surv = survival_prob[idx_after - 1]  # Step function
#                 se = np.sqrt(obs_surv * (1 - obs_surv) / count)

#             calib_obs.append(obs_surv)
#             calib_pred.append(avg_pred)
#             calib_counts.append(count)
#             calib_se.append(se)

#         except Exception as e:
#             print(f"Skipping bin {i}: {e}")
#             continue

#     # Convert to arrays
#     calib_pred = np.array(calib_pred)
#     calib_obs = np.array(calib_obs)
#     calib_se = np.array(calib_se)

#     # Plot with error bars
#     plt.figure(figsize=(7, 7))

#     # Perfect calibration
#     plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label="Perfect Calibration", alpha=0.7)

#     # Calibration curve with confidence intervals
#     plt.errorbar(calib_pred, calib_obs, yerr=1.96*calib_se,
#                  fmt='o-', capsize=5, capthick=2,
#                  label=f"Observed (n={len(preds_surv)})")

#     # Add sample size annotations
#     for pred, obs, count in zip(calib_pred, calib_obs, calib_counts):
#         plt.annotate(f'n={count}', (pred, obs),
#                     textcoords="offset points", xytext=(0,10),
#                     ha='center', fontsize=8, alpha=0.6)

#     plt.xlabel("Predicted Survival Probability", fontsize=12)
#     plt.ylabel("Observed Survival Probability (KM)", fontsize=12)
#     plt.title(f"Calibration Plot at t={time_point:.1f}", fontsize=14)
#     plt.legend(loc='best')
#     plt.grid(True, alpha=0.3)
#     plt.xlim(-0.05, 1.05)
#     plt.ylim(-0.05, 1.05)
#     plt.tight_layout()

#     # Calculate calibration metrics
#     ece = np.sum(calib_counts * np.abs(calib_pred - calib_obs)) / sum(calib_counts)
#     print(f"Expected Calibration Error (ECE): {ece:.4f}")

#     return calib_pred, calib_obs, ece

In [ ]:
# from lifelines import KaplanMeierFitter
# from scipy.stats import kstest

# def d_calibration(preds_surv, obs_times, obs_events, time_point):
#     """
#     D-calibration: Tests if distribution of predictions matches observed.
#     Uses probability integral transform (PIT).
#     """

#     # Compute PIT values
#     pit_values = []
#     for pred, time, event in zip(preds_surv, obs_times, obs_events):
#         if time < time_point:
#             # Event occurred before time_point
#             pit = pred if event else 1.0
#         else:
#             # Censored or survived past time_point
#             pit = pred
#         pit_values.append(pit)

#     pit_values = np.array(pit_values)

#     # If well-calibrated, PIT should be uniform [0,1]
#     ks_stat, p_value = kstest(pit_values, 'uniform')

#     print(f"KS test: statistic={ks_stat:.4f}, p-value={p_value:.4f}")

#     # Plot PIT histogram
#     plt.figure(figsize=(8, 5))
#     plt.hist(pit_values, bins=20, density=True, alpha=0.7, edgecolor='black')
#     plt.axhline(1.0, color='red', linestyle='--', label='Perfect calibration (uniform)')
#     plt.xlabel('PIT Values')
#     plt.ylabel('Density')
#     plt.title(f'D-Calibration at t={time_point:.1f}')
#     plt.legend()
#     plt.tight_layout()

#     return ks_stat, p_value

In [ ]:
# # Evaluate all treatment lines and time points
# N_LINES = 3  # Adjust to your number of lines

# for line_idx in range(N_LINES):
#     print(f"\n{'='*60}")
#     print(f"Treatment Line {line_idx + 1}")
#     print(f"{'='*60}")

#     # Create subplot figure for this line
#     n_timepoints = len(model.interval_bounds) - 2
#     fig, axes = plt.subplots(1, min(3, n_timepoints), figsize=(15, 5))
#     if n_timepoints == 1:
#         axes = [axes]

#     for tp_idx in range(1, min(4, len(model.interval_bounds)-1)):
#         # Gather data for this line and time point
#         preds_surv = []
#         obs_times = []
#         obs_events = []
#         time_val = model.interval_bounds[tp_idx].item()

#         for batch_res in all_predictions:
#             mask_line = batch_res["mask"][:, line_idx].astype(bool)
#             if not mask_line.any():
#                 continue
#             haz_logits = batch_res["hazards_logit"][mask_line, line_idx, :]
#             haz = 1 / (1 + np.exp(-haz_logits))
#             surv = np.cumprod(1 - haz, axis=1)
#             preds_surv.extend(surv[:, tp_idx])
#             obs_times.extend(batch_res["time"][mask_line, line_idx])
#             obs_events.extend(batch_res["event"][mask_line, line_idx])

#         preds_surv = np.array(preds_surv)
#         obs_times = np.array(obs_times)
#         obs_events = np.array(obs_events)

#         print(f"\nTime point {tp_idx} (t={time_val:.2f}):")

#         # Run calibration
#         calib_pred, calib_obs, ece = plot_calibration_improved(
#             preds_surv, obs_times, obs_events, time_val, n_bins=10
#         )

#         # Run D-calibration
#         ks_stat, p_value = d_calibration(
#             preds_surv, obs_times, obs_events, time_val
#         )

#     plt.show()